In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path = '/content/drive/MyDrive/DeepfakeData/SampleData/organized'

In [ ]:
import os

destination_base = '/content/drive/MyDrive/DeepfakeData/SampleData/frames'
subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

for subdir in subdirectories:
    full_path = os.path.join(destination_base, subdir)
    os.makedirs(full_path, exist_ok=True)

print("Directories created successfully.")

Directories created successfully.


In [ ]:
import os

video_files = []
main_subdirs = ['train', 'val', 'test']
inner_subdirs = ['real_videos', 'fake_videos']

for main_subdir in main_subdirs:
    main_subdir_path = os.path.join(data_path, main_subdir)
    if os.path.isdir(main_subdir_path):
        for inner_subdir in inner_subdirs:
            inner_subdir_path = os.path.join(main_subdir_path, inner_subdir)
            if os.path.isdir(inner_subdir_path):
                for filename in os.listdir(inner_subdir_path):
                    if filename.endswith('.mp4'):
                        video_files.append(os.path.join(inner_subdir_path, filename))

print(f"Found {len(video_files)} video files.")

Found 400 video files.


In [ ]:
# import subprocess
# import os

# destination_base = '/content/drive/MyDrive/DeepfakeData/SampleData/frames'

# for video_file in video_files:
#     relative_path = os.path.relpath(video_file, data_path)
#     output_dir = os.path.join(destination_base, os.path.dirname(relative_path), os.path.splitext(os.path.basename(video_file))[0])
#     os.makedirs(output_dir, exist_ok=True)
#     output_frame_pattern = os.path.join(output_dir, '%04d.png')

#     ffmpeg_command = [
#         'ffmpeg',
#         '-i', video_file,
#         '-vf', 'fps=3',
#         output_frame_pattern
#     ]

#     try:
#         subprocess.run(ffmpeg_command, check=True, capture_output=True, text=True)
#         print(f"Successfully extracted frames from {video_file}")
#     except subprocess.CalledProcessError as e:
#         print(f"Error extracting frames from {video_file}: {e}")
#         print(f"Stderr: {e.stderr}")
#     except FileNotFoundError:
#         print("Error: ffmpeg command not found. Make sure ffmpeg is installed and in your PATH.")

Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/aelfnikyqj.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/afoovlsmtx.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/agrmhtjdlk.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/ajqslcypsw.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/anpuvshzoo.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/asaxgevnnp.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/atkdltyyen.mp4
Successfully extracted frames from /content/drive/MyDrive/DeepfakeData/SampleData/organized/train/real_videos/atvmxvwyns.mp4


In [ ]:
# #!/usr/bin/env python3
# """
# crop_faces.py — Detect, 1.3x enlarge, and save 299x299 face crops from frame folders.
# Useful to prepare ImageFolder datasets for training.

# Input frames dir structure:
# frames/
#   videoA/frame_00001.jpg
#   videoA/frame_00002.jpg
#   ...
#   videoB/frame_00001.jpg
#   ...

# Usage:
# Run the crop_faces_from_video_folder function with appropriate arguments.
# """

# import argparse, os
# from pathlib import Path
# import cv2
# import mediapipe as mp

# def enlarge_box(x1,y1,x2,y2, scale, W, H):
#     cx, cy = (x1+x2)/2.0, (y1+y2)/2.0
#     w, h = (x2-x1)*scale, (y2-y1)*scale
#     nx1, ny1 = max(0, int(cx - w/2)), max(0, int(cy - h/2))
#     nx2, ny2 = min(W-1, int(cx + w/2)), min(H-1, int(cy + h/2))
#     return nx1, ny1, nx2, ny2

# def detect_largest_face_mediapipe(img, face_detector):
#     # Convert the image to RGB as mediapipe requires RGB input
#     img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     results = face_detector.process(img_rgb)

#     if not results.detections:
#         return None

#     largest_face = None
#     largest_area = 0

#     for detection in results.detections:
#         bbox = detection.location_data.relative_bounding_box
#         ih, iw, _ = img.shape
#         x, y, w, h = int(bbox.xmin * iw), int(bbox.ymin * ih), int(bbox.width * iw), int(bbox.height * ih)
#         area = w * h
#         if area > largest_area:
#             largest_area = area
#             largest_face = (x, y, x+w, y+h)

#     return largest_face


# def crop_faces_from_video_folder(frames_dir, out_dir, enlarge_scale=1.3, jpeg_quality=95):
#     """
#     Detect, enlarge, and save face crops from frame folders.

#     Args:
#         frames_dir (str): Directory with per-video subfolders of frames.
#         out_dir (str): Where to write 299x299 crops.
#         enlarge_scale (float, optional): Scale to enlarge the face bounding box. Defaults to 1.3.
#         jpeg_quality (int, optional): JPEG quality for saved images. Defaults to 95.
#     """
#     mp_face_detection = mp.solutions.face_detection
#     mp_drawing = mp.solutions.drawing_utils

#     frames_root = Path(frames_dir)
#     out_root = Path(out_dir)
#     out_root.mkdir(parents=True, exist_ok=True)

#     videos = sorted([p for p in frames_root.iterdir() if p.is_dir()])
#     count = 0

#     # Initialize Mediapipe Face Detection
#     with mp_face_detection.FaceDetection(
#         model_selection=1, min_detection_confidence=0.5) as face_detector:

#         for vdir in videos:
#             # Change glob pattern to look for .png files
#             for fp in sorted(vdir.glob("*.png")):
#                 img = cv2.imread(str(fp))
#                 if img is None:
#                     continue
#                 H, W = img.shape[:2]
#                 box = detect_largest_face_mediapipe(img, face_detector)
#                 if box is None:
#                     continue
#                 x1,y1,x2,y2 = box
#                 x1,y1,x2,y2 = enlarge_box(x1,y1,x2,y2, enlarge_scale, W, H)
#                 crop = img[y1:y2, x1:x2]
#                 crop = cv2.resize(crop, (299,299), interpolation=cv2.INTER_AREA)
#                 out_fp = out_root / f"{vdir.name}_{fp.stem}.jpg" # Save as .jpg as requested
#                 cv2.imwrite(str(out_fp), crop, [int(cv2.IMWRITE_JPEG_QUALITY), jpeg_quality])
#                 count += 1

#     print(f"Saved {count} crops to {out_root}")

# # Example usage (you can call this function in a separate cell):
# # frames_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/frames' # Replace with your frames directory
# # out_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/crops/train/real' # Replace with your desired output directory
# # crop_faces_from_video_folder(frames_dir, out_dir)

In [ ]:
import os

frames_base_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/frames'
subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

for subdir in subdirectories:
    frames_subdir = os.path.join(frames_base_dir, subdir)
    print(f"Checking directory: {frames_subdir}")
    if os.path.isdir(frames_subdir):
        print("Directory exists.")
        # Check for files within the directory (limit to first few for brevity)
        files_in_dir = os.listdir(frames_subdir)
        if files_in_dir:
            print(f"Found {len(files_in_dir)} items in directory. First 5: {files_in_dir[:5]}")
            # Further check if any expected image files exist within the subdirectories (video folders)
            video_subdirs = [d for d in os.listdir(frames_subdir) if os.path.isdir(os.path.join(frames_subdir, d))]
            if video_subdirs:
                print(f"Found {len(video_subdirs)} video subdirectories. Checking the first one...")
                first_video_subdir = video_subdirs[0]
                first_video_subdir_path = os.path.join(frames_subdir, first_video_subdir)
                frames_in_video_subdir = [f for f in os.listdir(first_video_subdir_path) if f.endswith('.jpg') or f.endswith('.png')]
                if frames_in_video_subdir:
                    print(f"Found {len(frames_in_video_subdir)} image files in {first_video_subdir}. First 5: {frames_in_video_subdir[:5]}")
                else:
                    print(f"No image files found in {first_video_subdir}.")
            else:
                print("No video subdirectories found within this directory.")
        else:
            print("Directory is empty.")
    else:
        print("Directory does not exist.")
    print("-" * 30)

Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/frames/train/real_videos
Directory exists.
Found 61 items in directory. First 5: ['aelfnikyqj', 'afoovlsmtx', 'agrmhtjdlk', 'ajqslcypsw', 'anpuvshzoo']
Found 61 video subdirectories. Checking the first one...
Found 30 image files in aelfnikyqj. First 5: ['0001.png', '0002.png', '0003.png', '0004.png', '0005.png']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/frames/train/fake_videos
Directory exists.
Found 258 items in directory. First 5: ['aagfhgtpmv', 'aapnvogymq', 'abofeumbvv', 'acifjvzvpm', 'aczrgyricp']
Found 258 video subdirectories. Checking the first one...
Found 30 image files in aagfhgtpmv. First 5: ['0001.png', '0002.png', '0003.png', '0004.png', '0005.png']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/frames/val/real_videos
Directory exists.
Found 8 items in directory. First 5: ['abarnvbtwb', 'avmjormvsx'

In [ ]:
# import os

# # Define the base directories
# frames_base_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/frames'
# crops_base_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/crops'

# # Define the subdirectories for frames and crops
# subdirectories = [
#     'train/real_videos',
#     'train/fake_videos',
#     'val/real_videos',
#     'val/fake_videos',
#     'test/real_videos',
#     'test/fake_videos'
# ]

# # Process each subdirectory
# for subdir in subdirectories:
#     frames_subdir = os.path.join(frames_base_dir, subdir)
#     crops_subdir = os.path.join(crops_base_dir, subdir)

#     # Ensure the output directory exists
#     os.makedirs(crops_subdir, exist_ok=True)

#     print(f"Processing frames from: {frames_subdir}")
#     print(f"Saving crops to: {crops_subdir}")

#     # Call the crop_faces_from_video_folder function
#     crop_faces_from_video_folder(frames_subdir, crops_subdir)
#     print("-" * 30)

# print("Face cropping process completed for all subdirectories.")

Processing frames from: /content/drive/MyDrive/DeepfakeData/SampleData/frames/train/real_videos
Saving crops to: /content/drive/MyDrive/DeepfakeData/SampleData/crops/train/real_videos
Saved 1789 crops to /content/drive/MyDrive/DeepfakeData/SampleData/crops/train/real_videos
------------------------------
Processing frames from: /content/drive/MyDrive/DeepfakeData/SampleData/frames/train/fake_videos
Saving crops to: /content/drive/MyDrive/DeepfakeData/SampleData/crops/train/fake_videos
Saved 7387 crops to /content/drive/MyDrive/DeepfakeData/SampleData/crops/train/fake_videos
------------------------------
Processing frames from: /content/drive/MyDrive/DeepfakeData/SampleData/frames/val/real_videos
Saving crops to: /content/drive/MyDrive/DeepfakeData/SampleData/crops/val/real_videos
Saved 240 crops to /content/drive/MyDrive/DeepfakeData/SampleData/crops/val/real_videos
------------------------------
Processing frames from: /content/drive/MyDrive/DeepfakeData/SampleData/frames/val/fake_vi

In [ ]:
import os

crops_base_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/crops'
subdirectories = [
    'train/real_videos',
    'train/fake_videos',
    'val/real_videos',
    'val/fake_videos',
    'test/real_videos',
    'test/fake_videos'
]

print("Checking if crop directories contain data:")

for subdir in subdirectories:
    crops_subdir = os.path.join(crops_base_dir, subdir)
    print(f"Checking directory: {crops_subdir}")
    if os.path.isdir(crops_subdir):
        files_in_dir = os.listdir(crops_subdir)
        if files_in_dir:
            print(f"Directory exists and contains {len(files_in_dir)} files. First 5: {files_in_dir[:5]}")
        else:
            print("Directory exists but is empty.")
    else:
        print("Directory does not exist.")
    print("-" * 30)

Checking if crop directories contain data:
Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/crops/train/real_videos
Directory exists and contains 1789 files. First 5: ['cfxkpiweqt_0015.jpg', 'cfxkpiweqt_0016.jpg', 'cfxkpiweqt_0017.jpg', 'cfxkpiweqt_0018.jpg', 'cfxkpiweqt_0019.jpg']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/crops/train/fake_videos
Directory exists and contains 7387 files. First 5: ['ecuvtoltue_0029.jpg', 'ecuvtoltue_0030.jpg', 'ecwaxgutkc_0001.jpg', 'ecwaxgutkc_0002.jpg', 'ecwaxgutkc_0003.jpg']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/crops/val/real_videos
Directory exists and contains 240 files. First 5: ['abarnvbtwb_0001.jpg', 'abarnvbtwb_0002.jpg', 'abarnvbtwb_0003.jpg', 'abarnvbtwb_0004.jpg', 'abarnvbtwb_0005.jpg']
------------------------------
Checking directory: /content/drive/MyDrive/DeepfakeData/SampleData/crops/val/fake_videos
Dire

In [ ]:
%pip install timm==0.6.13

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 549.1/549.1 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: timm
    Found existing installation: timm 1.0.22
    Uninstalling timm-1.0.22:
      Successfully uninstalled timm-1.0.22


In [ ]:
!pip install torch_xla

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 MB 31.3 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 155.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.3/39.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0

In [ ]:
#!/usr/bin/env python3
"""
train_xception.py — Fine-tune XceptionNet (via timm) on face crops (real/fake)
Dataset layout (ImageFolder style):
crops/
  train/real/*.jpg
  train/fake/*.jpg
  val/real/*.jpg
  val/fake/*.jpg

Usage:
Run the train_model function with appropriate arguments.
"""

import argparse, os, random, time, math
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Import torch_xla
# import torch_xla
# import torch_xla.core.xla_model as xm

try:
    import timm
except Exception as e:
    raise SystemExit("Please install timm: pip install timm==1.*") from e

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def build_loaders(data_dir, img_size=299, batch_size=64, num_workers=8):
    train_tf = transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.9, 1.0)),
        transforms.RandomApply([transforms.GaussianBlur(3)], p=0.2),
        transforms.ColorJitter(0.1,0.1,0.1,0.05),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    val_tf = transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_tf)
    val_ds   = datasets.ImageFolder(os.path.join(data_dir, "val"),   transform=val_tf)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, train_ds.classes

In [ ]:
def evaluate_frame_level(model, loader, device, alpha=1.0):
    """
    Evaluate model and compute metrics including weighted precision.

    Args:
        model: The model to evaluate
        loader: DataLoader for evaluation
        device: Device to run evaluation on
        alpha: Weight factor for false positives in weighted precision.
               alpha = (ratio of real:fake in organic traffic) / (ratio of real:fake in dataset)
               For example, if organic is 1M:1 and dataset is 1:1, alpha = 1M
               Default 1.0 means no weighting (standard precision)

    Returns:
        Dictionary of metrics including weighted precision (wP)
    """
    from sklearn.metrics import confusion_matrix

    model.eval()
    all_logits, all_targets = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            all_logits.append(logits.cpu())
            all_targets.append(y)
    logits = torch.cat(all_logits, dim=0)
    targets = torch.cat(all_targets, dim=0).numpy()
    probs = torch.softmax(logits, dim=1).numpy()[:,1]
    preds = (probs >= 0.5).astype(np.int64)

    # Calculate confusion matrix
    # Assuming: 0 = real, 1 = fake
    cm = confusion_matrix(targets, preds)
    if cm.shape == (2, 2):
        TN, FP, FN, TP = cm.ravel()
    else:
        # Handle edge cases where one class is missing
        if len(np.unique(targets)) == 1:
            if targets[0] == 0:  # All real
                TN, FP, FN, TP = len(targets), 0, 0, 0
            else:  # All fake
                TN, FP, FN, TP = 0, 0, 0, len(targets)
        else:
            TN, FP, FN, TP = 0, 0, 0, 0

    metrics = {}
    metrics["acc"] = float(accuracy_score(targets, preds))
    metrics["f1"]  = float(f1_score(targets, preds))

    # Calculate standard precision and recall
    if TP + FP > 0:
        metrics["precision"] = float(TP / (TP + FP))
    else:
        metrics["precision"] = 0.0

    if TP + FN > 0:
        metrics["recall"] = float(TP / (TP + FN))
    else:
        metrics["recall"] = 0.0

    # Calculate weighted precision: wP = TP / (TP + α * FP)
    if TP + alpha * FP > 0:
        metrics["wP"] = float(TP / (TP + alpha * FP))
    else:
        metrics["wP"] = 0.0

    # Additional metrics
    try:
        metrics["auc"] = float(roc_auc_score(targets, probs))
        metrics["ap_fake"] = float(average_precision_score(targets, probs))
    except Exception:
        metrics["auc"] = float("nan")
        metrics["ap_fake"] = float("nan")

    # Store confusion matrix components for debugging
    metrics["TP"] = int(TP)
    metrics["FP"] = int(FP)
    metrics["FN"] = int(FN)
    metrics["TN"] = int(TN)

    return metrics

In [ ]:
def save_checkpoint(state, is_best, out_dir, name="xception"):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt = out_dir / f"{name}_last_dec2.pt"
    torch.save(state, ckpt)
    if is_best:
        best = out_dir / f"{name}_best_dec2.pt"
        torch.save(state, best)

In [ ]:
def train_model(data_dir, out_dir, epochs_head=3, epochs_full=15, batch_size=64, img_size=299, num_workers=8, seed=42, lr_head=3e-4, lr_full=1e-4, weight_decay=1e-4, resume=True, alpha=1.0):
    """
    Train Xception model with checkpoint resuming and weighted precision for model selection.

    Args:
        alpha: Weight factor for false positives in weighted precision.
               alpha = (ratio of real:fake in organic traffic) / (ratio of real:fake in dataset)
               For example, if organic traffic has 1M real:1 fake and dataset has 1:1, alpha = 1,000,000
               Default 1.0 means standard precision (no weighting).
               Higher alpha penalizes false positives more, which is important for class imbalance.
    """
    seed_everything(seed)

    # Use CUDA device if available, otherwise CPU
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    print(f"Using weighted precision with alpha={alpha} for model selection")
    print(f"  (alpha = ratio of real:fake in organic traffic / ratio of real:fake in dataset)")
    print(f"  (higher alpha penalizes false positives more, important for class imbalance)")

    train_loader, val_loader, classes = build_loaders(data_dir, img_size, batch_size, num_workers)
    print(f"Classes: {classes} (expect ['fake', 'real'] or similar two classes)")

    # Check for existing checkpoint
    out_dir_path = Path(out_dir)
    checkpoint_path = out_dir_path / "xception_last_dec2.pt"
    start_epoch = 0
    best_val = -1.0
    resume_from_phase2 = False
    checkpoint = None

    if resume and checkpoint_path.exists():
        print(f"Found checkpoint at {checkpoint_path}. Loading...")
        checkpoint = torch.load(checkpoint_path, map_location=device)

        # Build model
        model = timm.create_model("xception", pretrained=False, num_classes=2)
        model.load_state_dict(checkpoint['model_state'])
        model = model.to(device)

        # Restore training state
        start_epoch = checkpoint.get('epoch', 0)
        # Restore best weighted precision, fallback to accuracy for old checkpoints
        val_metrics_ckpt = checkpoint.get('val_metrics', {})
        best_val = val_metrics_ckpt.get('wP', val_metrics_ckpt.get('acc', -1.0))

        # Check which phase we were in
        if start_epoch > epochs_head:
            # We were in Phase 2
            resume_from_phase2 = True
            print(f"Resuming from Phase 2, epoch {start_epoch - epochs_head}/{epochs_full} (total epoch {start_epoch})")
        else:
            # We were in Phase 1
            print(f"Resuming from Phase 1, epoch {start_epoch}/{epochs_head}")

        # Restore args if they exist (for validation)
        saved_args = checkpoint.get('args', {})
        if saved_args:
            print(f"Checkpoint was saved with args: epochs_head={saved_args.get('epochs_head')}, epochs_full={saved_args.get('epochs_full')}")
    else:
        # Build model from scratch
        print("No checkpoint found. Starting training from scratch with pretrained ImageNet weights.")
        model = timm.create_model("xception", pretrained=True, num_classes=2)
        model = model.to(device)

    # Phase 1: freeze backbone, train head
    head_params = []
    for n, p in model.named_parameters():
        if "fc" in n or "classifier" in n:
            head_params.append(p)
            p.requires_grad = True
        else:
            p.requires_grad = False

    opt = torch.optim.AdamW(head_params, lr=lr_head, weight_decay=weight_decay)
    crit = nn.CrossEntropyLoss()

    # Load optimizer state if resuming from Phase 1
    if resume and checkpoint is not None and not resume_from_phase2:
        if 'optimizer_state' in checkpoint:
            opt.load_state_dict(checkpoint['optimizer_state'])
            print("Loaded optimizer state from checkpoint.")

    def train_one_epoch(epoch, model, loader, optimizer, criterion, device):
        model.train()
        losses = []
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        return float(np.mean(losses))

    # Phase 1: freeze backbone, train head (skip if resuming from Phase 2)
    if not resume_from_phase2:
        print("==> Phase 1: training the head")
        phase1_start = start_epoch + 1 if start_epoch < epochs_head else epochs_head + 1
        for epoch in range(phase1_start, epochs_head+1):
            tr_loss = train_one_epoch(epoch, model, train_loader, opt, crit, device)
            val_metrics = evaluate_frame_level(model, val_loader, device, alpha=alpha)
            # Use weighted precision for model selection (better for class imbalance)
            score = val_metrics["wP"]
            is_best = score > best_val
            if is_best: best_val = score
            save_checkpoint({
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": opt.state_dict(),
                "val_metrics": val_metrics,
                "args": {
                    "data_dir": data_dir, "out_dir": out_dir, "epochs_head": epochs_head,
                    "epochs_full": epochs_full, "batch_size": batch_size, "img_size": img_size,
                    "num_workers": num_workers, "seed": seed, "lr_head": lr_head,
                    "lr_full": lr_full, "weight_decay": weight_decay, "alpha": alpha
                },
            }, is_best, out_dir, name="xception")
            print(f"[Head {epoch}/{epochs_head}] loss={tr_loss:.4f}  wP={score:.4f}  acc={val_metrics['acc']:.4f}  prec={val_metrics.get('precision', 0):.4f}  rec={val_metrics.get('recall', 0):.4f}  TP={val_metrics.get('TP', 0)} FP={val_metrics.get('FP', 0)}")
    else:
        print("==> Phase 1: already completed, skipping to Phase 2")

    # Phase 2: unfreeze all, train
    for p in model.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(model.parameters(), lr=lr_full, weight_decay=weight_decay)

    # Load optimizer state if resuming from Phase 2
    if resume and checkpoint is not None and resume_from_phase2:
        if 'optimizer_state' in checkpoint:
            opt.load_state_dict(checkpoint['optimizer_state'])
            print("Loaded optimizer state from checkpoint.")

    print("==> Phase 2: fine-tuning all layers")
    total_epochs = epochs_full

    # Calculate starting epoch for Phase 2
    if resume_from_phase2:
        phase2_start = (start_epoch - epochs_head) + 1
    else:
        phase2_start = 1

    for epoch in range(phase2_start, total_epochs+1):
        tr_loss = train_one_epoch(epoch, model, train_loader, opt, crit, device)
        val_metrics = evaluate_frame_level(model, val_loader, device, alpha=alpha)
        # Use weighted precision for model selection (better for class imbalance)
        score = val_metrics["wP"]
        is_best = score > best_val
        if is_best: best_val = score
        save_checkpoint({
            "epoch": epochs_head + epoch,
            "model_state": model.state_dict(),
            "optimizer_state": opt.state_dict(),
            "val_metrics": val_metrics,
            "args": {
                "data_dir": data_dir, "out_dir": out_dir, "epochs_head": epochs_head,
                "epochs_full": epochs_full, "batch_size": batch_size, "img_size": img_size,
                "num_workers": num_workers, "seed": seed, "lr_head": lr_head,
                "lr_full": lr_full, "weight_decay": weight_decay, "alpha": alpha
            },
        }, is_best, out_dir, name="xception")
        print(f"[FT {epoch}/{total_epochs}] loss={tr_loss:.4f}  wP={score:.4f}  acc={val_metrics['acc']:.4f}  prec={val_metrics.get('precision', 0):.4f}  rec={val_metrics.get('recall', 0):.4f}  TP={val_metrics.get('TP', 0)} FP={val_metrics.get('FP', 0)}")

    print(f"Training complete. Best weighted precision (wP): {best_val:.4f}")

# Example usage (you can call this function in a separate cell):
data_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/crops' # Replace with your crops directory
out_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/runs/xception_dfdc' # Replace with your desired output directory for runs

# Calculate alpha from your dataset if needed:
# For example, if your dataset has 1000 real:1000 fake (1:1 ratio) and organic traffic is 1M:1,
# then alpha = (1,000,000/1) / (1000/1000) = 1,000,000
# If unsure, start with alpha=1.0 (standard precision) or alpha=100-1000 for moderate imbalance

train_model(data_dir, out_dir, batch_size=32, alpha=1.0) # Reduced batch size to mitigate OOM
# For class imbalance, use higher alpha, e.g., alpha=1000 or alpha=10000

Using device: cuda
Using weighted precision with alpha=1.0 for model selection
  (alpha = ratio of real:fake in organic traffic / ratio of real:fake in dataset)
  (higher alpha penalizes false positives more, important for class imbalance)
Classes: ['fake_videos', 'real_videos'] (expect ['fake', 'real'] or similar two classes)
No checkpoint found. Starting training from scratch with pretrained ImageNet weights.
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
==> Phase 1: training the head
[Head 1/3] loss=0.4577  wP=0.6429  acc=0.7891  prec=0.6429  rec=0.0375  TP=9 FP=5
[Head 2/3] loss=0.4104  wP=0.7273  acc=0.7900  prec=0.7273  rec=0.0333  TP=8 FP=3
[Head 3/3] loss=0.3836  wP=0.7273  acc=0.8079  prec=0.7273  rec=0.1667  TP=40 FP=15
==> Phase 2: fine-tuning all layers
[FT 1/15] loss=0.1390  wP=0.9600  acc=0.9705  prec=0.9600  rec=0.9000  TP=216 FP=9
[FT 2/15

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

# Define the path to the best model checkpoint
out_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/runs/xception_dfdc' # Make sure this matches your training output directory
best_model_path = os.path.join(out_dir, 'xception_best.pt')

# Define the path to the test data
data_dir = '/content/drive/MyDrive/DeepfakeData/SampleData/crops' # Make sure this matches your crops directory
test_data_dir = os.path.join(data_dir, 'test')

# Check if the best model checkpoint exists
if not os.path.exists(best_model_path):
    print(f"Error: Best model checkpoint not found at {best_model_path}")
else:
    # Load the best model state
    checkpoint = torch.load(best_model_path)

    # Build the model architecture (needs to match the one used for training)
    import timm
    model = timm.create_model("xception", pretrained=False, num_classes=2) # pretrained=False because we are loading a trained model
    model.load_state_dict(checkpoint['model_state'])

    # Set the model to evaluation mode
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    print(f"Loaded best model from {best_model_path} and set to evaluation mode on {device}.")

    # Define the transformations for the test set (same as validation)
    img_size = 299
    test_tf = transforms.Compose([
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])

    # Create the test dataset and dataloader
    try:
        test_ds = datasets.ImageFolder(test_data_dir, transform=test_tf)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=8, pin_memory=True) # Use a reasonable batch size
        print(f"Created test dataset with {len(test_ds)} images.")

        # Evaluate the model on the test set
        print("Evaluating model on the test set...")
        test_metrics = evaluate_frame_level(model, test_loader, device)

        print("\nTest Set Metrics:")
        print(f"  Accuracy: {test_metrics['acc']:.4f}")
        print(f"  F1 Score: {test_metrics['f1']:.4f}")
        print(f"  AUC: {test_metrics['auc']:.4f}")
        print(f"  AP (Fake): {test_metrics['ap_fake']:.4f}")

    except FileNotFoundError:
        print(f"Error: Test data directory not found at {test_data_dir}. Please ensure the path is correct.")
    except Exception as e:
        print(f"An error occurred during test set evaluation: {e}")

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Loaded best model from /content/drive/MyDrive/DeepfakeData/SampleData/runs/xception_dfdc/xception_best.pt and set to evaluation mode on cuda.
Created test dataset with 1159 images.
Evaluating model on the test set...

Test Set Metrics:
  Accuracy: 0.9413
  F1 Score: 0.8687
  AUC: 0.9899
  AP (Fake): 0.9668
